# Music Generator — Google Colab Setup

Two processes run side by side:
- **Magenta service** (port 8001): Python 3.7 conda env with TF1 + Magenta.
- **Main API** (port 8000): System Python (3.10+) with AutoGen + Gemini.

Round 2 adds these optional layers — each in its own cell so you can skip any that
you don't want:
- Reference engines (MusicLang / Anticipatory / CA2 / MMT / FIGARO) feed the
  Composer alternative drafts alongside Magenta.
- FluidSynth + a bundled SoundFont render `.mid → .wav` so you can actually hear
  the output (no DAW needed).
- `pedalboard` mix bus applies per-section reverb / sidechain / panning.
- OpenUtau + DiffSinger synthesize Chinese vocals from the lyric plan (heavy;
  downloads a ~800 MB voicebank on first use — keep disabled if you just want
  instrumentals).

## Step 1: Clone repo

In [ ]:
# If private repo: !git clone https://<TOKEN>@github.com/Jefferson8868/AI_Music.git
!git clone https://github.com/Jefferson8868/AI_Music.git
%cd AI_Music

## Step 2: Install Miniconda + Python 3.7 env for Magenta

In [ ]:
%%bash
set -e
if [ ! -f /content/miniconda/bin/conda ]; then
  wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O /tmp/miniconda.sh
  bash /tmp/miniconda.sh -b -p /content/miniconda
  echo 'Miniconda installed'
else
  echo 'Miniconda already installed'
fi
CONDA=/content/miniconda/bin/conda
$CONDA tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main 2>/dev/null || true
$CONDA tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r 2>/dev/null || true
echo 'Conda ToS accepted'

In [ ]:
%%bash
set -e
CONDA=/content/miniconda/bin/conda
if [ ! -d /content/miniconda/envs/magenta_env ]; then
  $CONDA create -n magenta_env python=3.7 -y -q
fi
PY=/content/miniconda/envs/magenta_env/bin/python
if [ ! -f $PY ]; then
  echo 'ERROR: Python not found, recreating...'
  $CONDA env remove -n magenta_env -y 2>/dev/null || true
  $CONDA create -n magenta_env python=3.7 -y
fi
$PY --version

In [ ]:
# System build deps needed by python-rtmidi (Magenta dependency).
!apt-get install -y -qq libasound2-dev libjack-dev build-essential > /dev/null 2>&1
print('System build deps installed')

# Magenta + web server in the Python 3.7 env.
!/content/miniconda/envs/magenta_env/bin/pip install -q magenta "fastapi<0.100" "uvicorn[standard]<0.24"
print('Magenta + FastAPI installed in magenta_env')

## Step 3: Download Magenta models

In [ ]:
%%bash
mkdir -p models
[ ! -f models/attention_rnn.mag ] && wget -q http://download.magenta.tensorflow.org/models/attention_rnn.mag -P models/
[ ! -f models/polyphony_rnn.mag ] && wget -q http://download.magenta.tensorflow.org/models/polyphony_rnn.mag -P models/
ls -lh models/*.mag

## Step 4: Install main app dependencies (system Python)

In [ ]:
!pip install -q autogen-core autogen-agentchat "autogen-ext[openai]"
!pip install -q fastapi "uvicorn[standard]" "pydantic>=2.0" pydantic-settings \
    httpx music21 mido python-multipart websockets loguru \
    openai anthropic google-generativeai

## Step 5: (Optional) Round 2 reference engines

Each cell here installs **one** reference engine on top of Magenta. The factory is
fanout-tolerant — any engine not installed silently falls back to `NullEngine`, so
you can pick-and-choose. The comma-separated list of engines Magenta + the
installed ones go into the `MG_REFERENCE_ENGINES` env var in Step 7.

The engines used by `src/engine/factory.py`:
- `musiclang` — pip-installable, chord-conditioned symbolic model (BSD-3).
- `anticipatory` — pip-installable, infilling transformer (Apache-2.0).
- `composers_assistant` — clone + PYTHONPATH, multi-track MIDI infilling (MIT).
- `mmt` — clone + PYTHONPATH, multi-track transformer (MIT).
- `figaro` — clone + PYTHONPATH, description-conditioned model (MIT).

### 5a. MusicLang Predict (pip, lightweight — recommended)

In [ ]:
!pip install -q musiclang_predict
import importlib, importlib.util
print('musiclang_predict:', 'ok' if importlib.util.find_spec('musiclang_predict') else 'FAILED')

### 5b. Anticipatory Music Transformer (pip, lightweight — recommended)

In [ ]:
!pip install -q anticipation
import importlib.util
print('anticipation:', 'ok' if importlib.util.find_spec('anticipation') else 'FAILED')

### 5c. Composer's Assistant 2 (git clone, heavier — downloads T5 weights)

No pip distribution exists; clone the repo onto `PYTHONPATH` and let the wrapper
auto-detect it. Skip this cell to fall back to NullEngine for this engine.

In [ ]:
%%bash
set -e
mkdir -p /content/ref_engines
cd /content/ref_engines
if [ ! -d composers-assistant-REAPER ]; then
  git clone --depth 1 https://github.com/m-malandro/composers-assistant-REAPER.git
fi
# Expose as importable package name the wrapper tries first.
if [ ! -d /content/ref_engines/composers_assistant2 ]; then
  ln -s /content/ref_engines/composers-assistant-REAPER /content/ref_engines/composers_assistant2
fi
echo 'CA2 cloned at /content/ref_engines/composers-assistant-REAPER'

In [ ]:
# Model-side deps (T5 via Transformers). torch usually pre-installed on Colab.
!pip install -q "transformers>=4.30" "sentencepiece" "accelerate"
import sys; sys.path.insert(0, '/content/ref_engines')
import importlib.util
print('composers_assistant2 importable:',
      'ok' if importlib.util.find_spec('composers_assistant2') else 'FAILED')

### 5d. MMT / MMT-BERT (git clone, heavier — requires PyTorch + MuSPy)

In [ ]:
%%bash
set -e
mkdir -p /content/ref_engines
cd /content/ref_engines
if [ ! -d mmt ]; then
  git clone --depth 1 https://github.com/salu133445/mmt.git
fi
echo 'MMT cloned at /content/ref_engines/mmt'

In [ ]:
# MMT deps per its requirements.txt (only the pieces our wrapper touches).
!pip install -q muspy "numpy<2" tqdm pretty_midi
import sys; sys.path.insert(0, '/content/ref_engines/mmt')
import importlib.util
print('mmt importable:', 'ok' if importlib.util.find_spec('mmt') else 'FAILED')

### 5e. FIGARO (git clone, heavier — requires PyTorch + miditoolkit)

In [ ]:
%%bash
set -e
mkdir -p /content/ref_engines
cd /content/ref_engines
if [ ! -d figaro ]; then
  git clone --depth 1 https://github.com/dvruette/figaro.git
fi
echo 'FIGARO cloned at /content/ref_engines/figaro'

In [ ]:
!pip install -q miditoolkit pytorch_lightning
import sys; sys.path.insert(0, '/content/ref_engines/figaro')
import importlib.util
print('figaro importable:', 'ok' if importlib.util.find_spec('figaro') else 'FAILED')

### 5f. Make the reference clones visible to the API server

The server spawned in Step 8 inherits `os.environ`. Adding the clone parent to
`PYTHONPATH` here ensures the factory's lazy imports resolve when the pipeline
runs.

In [ ]:
import os
extras = ['/content/ref_engines',
          '/content/ref_engines/mmt',
          '/content/ref_engines/figaro']
cur = os.environ.get('PYTHONPATH', '')
os.environ['PYTHONPATH'] = ':'.join([p for p in extras if os.path.isdir(p)] +
                                    ([cur] if cur else []))
print('PYTHONPATH:', os.environ['PYTHONPATH'])

## Step 6: Audio render (FluidSynth + SoundFont)

Produces `{title}.wav` alongside `{title}.mid`. Recommended — Round 2 assumes you
can actually hear the output. If skipped, set `MG_RENDER_AUDIO=false` in Step 7.

In [ ]:
# FluidSynth system binary + Python binding + GM SoundFont fallback.
!apt-get install -y -qq fluidsynth fluid-soundfont-gm libsndfile1 > /dev/null 2>&1
!pip install -q pyFluidSynth soundfile numpy
import shutil, importlib.util
print('FluidSynth system:', shutil.which('fluidsynth') or 'NOT FOUND')
print('pyfluidsynth:', 'ok' if importlib.util.find_spec('fluidsynth') else 'FAILED')

In [ ]:
# Bundled SoundFont. If assets/soundfonts/combined.sf2 is in the repo, use it.
# Otherwise fall back to the GM bank that ships with apt.
import os
from pathlib import Path

combined = Path('assets/soundfonts/combined.sf2')
gm_fallback = Path('/usr/share/sounds/sf2/FluidR3_GM.sf2')

if combined.is_file():
    sf_path = combined.resolve()
    print(f'Using bundled SoundFont: {sf_path} ({sf_path.stat().st_size/1e6:.1f} MB)')
elif gm_fallback.is_file():
    sf_path = gm_fallback
    print(f'Bundled SoundFont missing. Using GM fallback: {sf_path}')
    print('For Chinese instruments, drop a merged bank at assets/soundfonts/combined.sf2.')
else:
    sf_path = None
    print('No SoundFont found — audio render will no-op.')

os.environ['MG_SOUNDFONT_PATH'] = str(sf_path) if sf_path else ''

## Step 7: Mix bus (pedalboard)

Per-section reverb, sidechain ducking, panning, bus compressor. Runs after the
audio render. If skipped, set `MG_APPLY_MIX=false`.

In [ ]:
!pip install -q pedalboard
import importlib.util
print('pedalboard:', 'ok' if importlib.util.find_spec('pedalboard') else 'FAILED')

## Step 8: (Optional) Vocal synthesis (OpenUtau + DiffSinger)

HEAVY: downloads an ~800 MB Chinese DiffSinger voicebank on first use and
requires .NET runtime for the OpenUtau CLI. Skip this cell if you want
instrumentals only — set `MG_SYNTHESIZE_VOCALS=false` in Step 9.

In [ ]:
# Chinese lyric → phoneme mapping (light).
!pip install -q pypinyin
import importlib.util
print('pypinyin:', 'ok' if importlib.util.find_spec('pypinyin') else 'FAILED')

In [ ]:
%%bash
# OpenUtau CLI — static Linux build. Runs headless with `OpenUtau.Cli --render`.
set -e
apt-get install -y -qq libicu-dev libssl-dev > /dev/null 2>&1
CACHE=/root/.cache/music_generator/openutau
mkdir -p $CACHE
cd $CACHE
if [ ! -f OpenUtau ]; then
  # Check https://github.com/stakira/OpenUtau/releases for newer builds.
  URL=https://github.com/stakira/OpenUtau/releases/download/build%2Flatest/OpenUtau-linux-x64.tar.gz
  wget -q "$URL" -O openutau.tgz || echo 'OpenUtau download failed'
  [ -f openutau.tgz ] && tar -xzf openutau.tgz && rm openutau.tgz
fi
ls -la $CACHE | head -20

In [ ]:
# Voicebank: pick ONE of the free Chinese DiffSinger banks (~800MB each).
# Example: 'Qixuan' from openvpi. Drop the unpacked folder under
# assets/voicebanks/ and point MG_VOCAL_VOICEBANK at it.
#
# Illustrative (commented out — large download + you should pick which bank):
# !mkdir -p assets/voicebanks && cd assets/voicebanks && \
#   wget -q https://github.com/openvpi/CHINESE-VOICEBANK-URL.zip && \
#   unzip -q CHINESE-VOICEBANK-URL.zip

import os
os.environ.setdefault('MG_VOCAL_VOICEBANK', 'default')
print('Vocal voicebank:', os.environ['MG_VOCAL_VOICEBANK'])
print('NOTE: Set MG_VOCAL_VOICEBANK to a real path before enabling vocals,')
print('      or the pipeline will log a warning and proceed without a vocal stem.')

## Step 9: Configure env vars

Add a Colab Secret named `GEMINI_API_KEY` (click the key icon in the left sidebar).
Get a free key at https://aistudio.google.com/apikey

`MG_REFERENCE_ENGINES` is the comma-separated fanout for the Composer. Keep
Magenta as the baseline and add any you actually installed above.

In [ ]:
import os
from google.colab import userdata

GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')

os.environ['MG_LLM_BACKEND'] = 'gemini'
os.environ['MG_LLM_MODEL'] = 'gemini-2.0-flash'
os.environ['MG_LLM_API_KEY'] = GEMINI_API_KEY

# Baseline engine (single-track Magenta service on :8001).
os.environ['MG_MUSIC_ENGINE'] = 'magenta'
os.environ['MG_MAGENTA_URL'] = 'http://localhost:8001'

# Round 2 Phase A: reference-engine fanout. Comma-separated list; any engine
# that isn't installed degrades to NullEngine and is silently dropped. Trim the
# list to what you actually installed in Step 5.
os.environ['MG_REFERENCE_ENGINES'] = ','.join([
    'magenta',
    'musiclang',       # 5a
    'anticipatory',    # 5b
    # 'composers_assistant',  # 5c (uncomment if cloned)
    # 'mmt',                  # 5d
    # 'figaro',               # 5e
])

# Round 2 Phase D: audio render.
os.environ['MG_RENDER_AUDIO'] = 'true'
# MG_SOUNDFONT_PATH already set in Step 6.
os.environ.setdefault('MG_RENDER_SAMPLE_RATE', '44100')

# Round 2 Phase E: vocals. Set to 'false' if you skipped Step 8.
os.environ['MG_SYNTHESIZE_VOCALS'] = 'false'

# Round 2 Phase F: mix bus.
os.environ['MG_APPLY_MIX'] = 'true'

# API server + pipeline tunables.
os.environ['MG_HOST'] = '0.0.0.0'
os.environ['MG_PORT'] = '8000'
os.environ['MG_MAX_GROUP_CHAT_MESSAGES'] = '50'
os.environ['MG_MAX_REFINEMENT_ROUNDS'] = '3'
os.environ['MG_CRITIC_PASS_THRESHOLD'] = '0.8'
os.environ['MG_QUANTIZATION_GRID'] = '0.25'

# Write .env fallback (read by pydantic-settings if env inheritance fails).
with open('.env', 'w') as f:
    for k, v in os.environ.items():
        if k.startswith('MG_'):
            f.write(f'{k}={v}\n')

print(f'LLM backend = {os.environ["MG_LLM_BACKEND"]}/{os.environ["MG_LLM_MODEL"]}')
print(f'Reference engines = {os.environ["MG_REFERENCE_ENGINES"]}')
print(f'Render audio = {os.environ["MG_RENDER_AUDIO"]}, '
      f'SF = {os.environ.get("MG_SOUNDFONT_PATH", "<none>")}')
print(f'Vocals = {os.environ["MG_SYNTHESIZE_VOCALS"]}, '
      f'Mix = {os.environ["MG_APPLY_MIX"]}')

## Step 10: Start Magenta service (Python 3.7)

In [ ]:
import subprocess, time, requests, shutil, os

MAGENTA_PYTHON = '/content/miniconda/envs/magenta_env/bin/python'
assert os.path.isfile(MAGENTA_PYTHON), f'Not found: {MAGENTA_PYTHON}'

shutil.copy('src/engine/magenta_service.py', '/content/magenta_app.py')

magenta_log = open('/content/magenta.log', 'w')
magenta_proc = subprocess.Popen(
    [MAGENTA_PYTHON, '-m', 'uvicorn', 'magenta_app:app',
     '--host', '0.0.0.0', '--port', '8001'],
    cwd='/content',
    env={**os.environ,
         'MELODY_MODEL': '/content/AI_Music/models/attention_rnn.mag',
         'POLY_MODEL': '/content/AI_Music/models/polyphony_rnn.mag'},
    stdout=magenta_log, stderr=subprocess.STDOUT
)

print(f'Magenta started (pid={magenta_proc.pid})')
for i in range(30):
    time.sleep(2)
    if magenta_proc.poll() is not None:
        print(f'Magenta exited with code {magenta_proc.returncode}')
        with open('/content/magenta.log') as f:
            print(f.read()[-3000:])
        break
    try:
        r = requests.get('http://localhost:8001/health', timeout=3)
        print(f'Magenta ready: {r.json()}')
        break
    except Exception:
        print(f'  waiting... ({(i+1)*2}s)', end='\r')
else:
    print('Magenta timed out. Log:')
    with open('/content/magenta.log') as f:
        print(f.read()[-3000:])

### (Debug) Check Magenta log

In [ ]:
with open('/content/magenta.log') as f:
    print(f.read()[-5000:])
print(f'Process alive: {magenta_proc.poll() is None}')

## Step 11: Start main API server

In [ ]:
import subprocess, time, requests, os

server = subprocess.Popen(
    ['python', '-m', 'uvicorn', 'src.main:app', '--host', '0.0.0.0', '--port', '8000'],
    cwd='/content/AI_Music',
    env=os.environ.copy(),
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
)

for i in range(15):
    time.sleep(1)
    try:
        r = requests.get('http://localhost:8000/api/health', timeout=2)
        print(f'Main server ready: {r.json()}')
        break
    except Exception:
        print(f'  waiting... ({i+1}s)', end='\r')
else:
    print('Main server failed:')
    print(server.stdout.read1(4096).decode())

## Step 12: Generate music

In [ ]:
import requests, json

print('Generating music (per-section composer, may take 3-8 minutes)...\n')

response = requests.post('http://localhost:8000/api/generate', json={
    'request': {
        'description': 'A gentle Chinese-style piano piece with pentatonic melody',
        'genre': 'classical',
        'mood': 'peaceful',
        'tempo': 80,
        'key': 'C',
        'scale_type': 'chinese_pentatonic',
        'sections': ['intro', 'verse', 'chorus', 'bridge', 'outro'],
        'bars_per_section': 4,
    }
}, timeout=1200)

result = response.json()
summary = result.get('summary', {})
midi_path = summary.get('midi_path')
wav_path = summary.get('wav_path')
snapshots = summary.get('snapshots', [])

print(f"Status: {result['status']}")
print(f"Final MIDI: {midi_path or 'none'}")
print(f"Final WAV:  {wav_path or 'none'}")

if snapshots:
    print(f"\n--- Round Snapshots ({len(snapshots)}) ---")
    for i, snap in enumerate(snapshots):
        label = 'Magenta draft' if 'magenta' in snap else f'Round {i}'
        print(f"  [{label}] {snap}")

print(f"\nSummary: {json.dumps({k:v for k,v in summary.items() if k != 'snapshots'}, indent=2)}")
print(f"\n--- Agent Messages ({len(result.get('messages', []))}) ---")
for i, msg in enumerate(result.get('messages', [])):
    preview = msg['content'][:200].replace('\n', ' ')
    print(f"\n[{i+1}] {msg['source']}: {preview}...")

## Step 13: Play + download

Prefers the `.wav` emitted by Step 6. Falls back to rendering the MIDI on the fly
via `fluidsynth` CLI if for some reason the pipeline didn't render one.

In [ ]:
import glob, os
from IPython.display import Audio, display

wav_files  = sorted(glob.glob('output/**/*.wav', recursive=True), key=os.path.getmtime, reverse=True)
midi_files = sorted(glob.glob('output/**/*.mid', recursive=True), key=os.path.getmtime, reverse=True)

if wav_files:
    print(f'Found {len(wav_files)} WAV files (using newest):')
    for f in wav_files[:5]:
        print(f'  {f} ({os.path.getsize(f)} bytes)')
    display(Audio(wav_files[0]))
elif midi_files:
    print('No WAV render found — rendering on the fly from MIDI.')
    midi_path = midi_files[0]
    wav_path = midi_path.replace('.mid', '.wav')
    sf = os.environ.get('MG_SOUNDFONT_PATH', '/usr/share/sounds/sf2/FluidR3_GM.sf2')
    !fluidsynth -ni "{sf}" "{midi_path}" -F "{wav_path}" -r 44100 > /dev/null 2>&1
    if os.path.exists(wav_path):
        display(Audio(wav_path))
else:
    print('No MIDI or WAV files. Check agent messages above.')

In [ ]:
from google.colab import files
for f in (wav_files[:1] + midi_files[:1]):
    files.download(f)

## Cleanup

In [ ]:
magenta_proc.terminate()
server.terminate()
print('Servers stopped.')